In [1]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter 
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
import time

# 1. Load API Key
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
os.environ["GOOGLE_API_KEY"] = api_key

c:\Users\PC\anaconda3\envs\llm\lib\site-packages\google\api_core\_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
c:\Users\PC\anaconda3\envs\llm\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def ingest_pdf_safe(file_path):
    print(f"--- Memproses file: {file_path} ---")

    try:
        # 1. Load PDF
        loader = PyPDFLoader(file_path)
        pages = loader.load()

        # 2. Split Teks (Menggunakan Recursive agar lebih cerdas membagi paragraf)
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=100,
            separators=["\n\n", "\n", " ", ""]
        )
        chunks = text_splitter.split_documents(pages)
        print(f"Berhasil memecah menjadi {len(chunks)} bagian.")
        
        # 3. Inisialisasi Embedding
        embeddings_model = GoogleGenerativeAIEmbeddings(
            model="models/gemini-embedding-2",
            task_type="retrieval_document"
        )

        # 4. EKSTRAK VEKTOR SECARA MANUAL
        print("Mengekstrak vektor satu per satu (Manual Mode)...")
        embedded_texts = []
        for i, doc in enumerate(chunks):
            # Ambil embedding untuk satu teks saja tiap iterasi
            vec = embeddings_model.embed_query(doc.page_content)
            # Simpan teks dan vektornya dalam tuple
            embedded_texts.append((doc.page_content, vec))
            print(f"Berhasil memproses bagian ke-{i+1}")
            # Opsional: beri jeda sebentar agar tidak terkena limit API
            time.sleep(0.5)

        # 5. MASUKKAN KE FAISS
        print("Membangun Vector Database...")
        # Kita gunakan form_embeddings agar FAISS tidak perlu memanggil API lagi
        vector_db = FAISS.from_embeddings(
            text_embeddings=embedded_texts,
            embedding=embeddings_model,
            metadatas=[doc.metadata for doc in chunks]
        )

        # 6. Simpan ke Lokal
        vector_db.save_local("faiss_index_basket")
        print("--- BERHASIL! Folder 'faiss_index_basket' telah tercipta ---")

    except Exception as e:
        print(f"Terjadi kendala: {e}")

if __name__ == "__main__":
    FILE_PDF = "data/basket-rules1.pdf"
    if os.path.exists(FILE_PDF):
        ingest_pdf_safe(FILE_PDF)
    else:
        print(f"File {FILE_PDF} tidak ditemukan. Pastikan file ada di folder yang sama dengan script.")

--- Memproses file: basket-rules1.pdf ---
Berhasil memecah menjadi 24 bagian.
Mengekstrak vektor satu per satu (Manual Mode)...
Berhasil memproses bagian ke-1
Berhasil memproses bagian ke-2
Berhasil memproses bagian ke-3
Berhasil memproses bagian ke-4
Berhasil memproses bagian ke-5
Berhasil memproses bagian ke-6
Berhasil memproses bagian ke-7
Berhasil memproses bagian ke-8
Berhasil memproses bagian ke-9
Berhasil memproses bagian ke-10
Berhasil memproses bagian ke-11
Berhasil memproses bagian ke-12
Berhasil memproses bagian ke-13
Berhasil memproses bagian ke-14
Berhasil memproses bagian ke-15
Berhasil memproses bagian ke-16
Berhasil memproses bagian ke-17
Berhasil memproses bagian ke-18
Berhasil memproses bagian ke-19
Berhasil memproses bagian ke-20
Berhasil memproses bagian ke-21
Berhasil memproses bagian ke-22
Berhasil memproses bagian ke-23
Berhasil memproses bagian ke-24
Membangun Vector Database...
--- BERHASIL! Folder 'faiss_index_basket' telah tercipta ---
